# 05 · 換地方:按**街道**取數據
想把整套流程搬到上海別的地方?**換地方 = 換一個街道。** 這本專講這件事。

> 和新加坡 OSM 版最根本的不同:OSM 版用「中心點 + 半徑」框一個**方塊**;
> 上海版的研究單位是**街道**(鄉鎮街道層的真實多邊形)——**非方形、大小不一**,
> 因爲形態家族(里弄 / 工人新村 / CBD)本就是按街道沉澱的。按街道取,纔對得上「誰更有權」的真實治理單位。

## 怎麼執行
- 點一格,按 **Shift+Enter** 執行;或選單 Run → Run All Cells 從頭跑。
- 結果與圖會顯示在該格下面。代碼都在 `engine/` 文件夾,平時不用打開。

In [ ]:
# 讓 notebook 找到 engine 裏的代碼(這格不用改,直接執行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就緒 ·  當前站點 SLUG =", config.SLUG, "(", config.site_name(), ")")

## 一、隨包就帶 9 個街道(跨 3 個形態家族)—— 離線就能切
下面這些街道的緩存隨包附帶,**改 `config.SLUG` 就能立刻切換**,不需要數據集、不需要聯網。

In [ ]:
import yaml
from pathlib import Path
print("隨包帶緩存的街道(改 config.SLUG 即可切):\n")
for s in config.SITES:
    p = C.DATA / s["slug"] / "buildings.parquet"
    mark = "✓緩存在" if p.exists() else "·需建"
    print("  %-10s %-7s %-12s %s" % (s["slug"], mark, s["name"], s["family"]))
print("\n默認主冊/報告跑這 3 個代表:", config.REPORT_SITES)

## 二、換成已有緩存的街道(最簡單)
**兩種改法,效果一樣:**
- 長期:打開 `config.py` 把 `SLUG = "..."` 改掉、存檔,回到上面「準備」格重跑。
- 臨時:在本 notebook 裏直接設 `config.SLUG`(下面這格示範切到外灘),馬上就能用。

In [ ]:
config.SLUG = "waitan"          # ✏️ 換成 config.SITES 裏任一 slug(如 nanjingxi / pengpu / laoximen …)
df = C.current_buildings(config.SLUG)
df = C.assign_all(df)
h = df["height_m"].astype(float)
print("現在是:%s(%s)" % (config.site_name(), config.SLUG))
print("  %d 棟 · 最高 %.0fm · 中位 %.0fm" % (len(df), h.max(), h.median()))
print("  角色:", df.stakeholder.value_counts().to_dict())
# 接着就能跑主冊 step1–5 / 進階算子,全部對這個新街道生效。

## 三、取一個**全新**街道(這 9 個之外)—— 需要數據集
想要隨包 9 個之外的街道(例如你研究的某個街道),就從原始「上海城市數據集」按**街道名**現建緩存。
需要先在 `config.py` 設 `DATASET_ROOT`(見 `數據集說明.md`)。下面這格:**列出數據集裏所有街道名**,方便你挑。

In [ ]:
P = C.dataset_paths()
if P["JD"].exists():
    import geopandas as gpd
    jd = gpd.read_file(P["JD"])
    names = sorted(n for n in jd["name"].dropna().unique())
    print("數據集裏共 %d 個鄉鎮街道。含『新村』的(工人新村家族)示例:" % len(names))
    print("  ", [n for n in names if "新村" in n][:12])
    print("含『路』的示例:", [n for n in names if n.endswith("路街道")][:8])
else:
    print("未設 DATASET_ROOT(或數據集不在)→ 這步需要數據集。")
    print("→ 沒有數據集也沒關係:上面隨包 9 個街道足夠練全套流程。")
    print("→ 想要別的街道:下載數據集、在 config.py 設 DATASET_ROOT,見 數據集說明.md。")

## 四、建這個新街道的緩存
選好街道名(鄉鎮街道層的精確名,支持「包含匹配」),給它一個英文 `slug`(輸出標識),建緩存。
沒有數據集時這格會自動跳過、提示用隨包緩存。建好後,它就和隨包街道一樣可離線用。

In [ ]:
NEW_NAME = "天山路街道"      # ✏️ 你要的街道名(数据集里的精确名;可用包含匹配)
NEW_SLUG = "tianshan"        # ✏️ 给它一个英文 slug(决定 data/<slug>/ 与 out/<slug>/)
if config.DATASET_ROOT and C.dataset_paths()["AI"].exists():
    C.build_cache(NEW_NAME, NEW_SLUG)        # 裁切 + 多源 join → data/<slug>/buildings.parquet
    config.SLUG = NEW_SLUG
    print("建好并切到:", config.site_name(NEW_SLUG))
else:
    print("没有数据集 → 跳过建缓存。先用随包的 9 个街道即可(见第一格)。")

## 五、(可選)寫進 config.py 長期用
臨時改 `config.SLUG` 只在本次有效。想長期用、並讓它進 `SITES` 菜單,在 `config.py` 的 `SITES` 加一行:
```python
SITES = [
    ...,
    {"slug": "tianshan", "name": "天山路街道", "family": "你給的家族標籤"},
]
```
要讓 `build_report.py` 也出它的報告,把 slug 加進 `REPORT_SITES`。

## 誠實邊界 & 小結
- 研究單位是**街道**(行政/治理單位),非方框、非產權邊界;一個街道一個 slug。
- 街道名用鄉鎮街道層的精確名(支持包含匹配,如 `subdistrict("天山")`)。
- 換地方後,回 [`03 主冊`] / [`04 進階`] 用新街道**重跑**即可——同一套權力邏輯,長在不同的真實基底上。
> 這正是進階冊的點題:**權力的幾何邏輯對基底不變,但結果在乎基底**。換地方,就是去驗證這句話。